# <font color='#1B3A5C'>Modelos de Deep Learning: LSTM y GRU multiserie</font>

> Tercer escalon de la progresion de modelado, tras el baseline estadistico (SARIMA/SARIMAX, notebook 05) y el gradient boosting global (LightGBM y XGBoost, notebooks 06 y 07). Se entrena una red recurrente unica sobre las 30 series limpias a la vez (un solo modelo global multiserie), con el mismo split temporal, los mismos horizontes y el mismo arnes de backtesting que el resto del estudio, de modo que la comparacion sea directa.

> Como variables exogenas se usan el calendario (festivo, hora, dia de la semana, etc.) y el clima a nivel de ciudad, ambos compartidos por todos los barrios. La arquitectura recurrente de skforecast admite exogenas comunes a todas las series, no una distinta por barrio; por eso el clima entra en su version agregada de ciudad (promedio entre barrios) y no por estacion. El clima fino por barrio y el perfil sectorial se reservan para un modelo por codigo postal sobre los barrios mas dificiles, al final del notebook.

> La comparacion con los arboles es directa en lo que importa (mismo target, CPs, split, horizontes, backtest y metricas). Lo unico que cambia es la representacion de las variables, donde cada familia de modelos usa su forma natural.

### Que hace este notebook
1. Backend de deep learning y diagnostico del entorno (Keras 3, CPU/GPU).
2. Carga del dataset y serie ancha (una columna por barrio) para la red recurrente.
3. Exogenas de ciudad (calendario compartido + clima promediado) y auditoria de nulos.
4. Imputacion de NaN (las redes no toleran NaN, a diferencia de los arboles) y escalado.
5. Modelo base LSTM, backtesting en validacion y control de sobreajuste.
6. Ajuste de hiperparametros y variante GRU.
7. Evaluacion en test y comparacion con el resto de familias.
8. Estudio dirigido por barrio (plan B) sobre los codigos mas dificiles.

---
# <font color='#1B3A5C'>0. Backend de deep learning y diagnóstico</font>

> `skforecast 0.22` apoya su `ForecasterRnn` en Keras 3, que necesita un backend de cálculo (TensorFlow, PyTorch o JAX). El contenedor no trae ninguno, así que se instala. Se usa la versión de CPU porque el contenedor no expone la GPU del equipo (el `docker-compose` no la pasa), y el volumen de datos no la requiere.


In [ ]:
# Ejecutar UNA sola vez y luego comentar esta celda (no hace falta reinstalar en cada arranque).
# TensorFlow es el backend mejor probado con skforecast. Versión CPU.
# Si pip intenta DEGRADAR numpy o muestra conflictos de dependencias, parar y avisar:
# en ese caso cambiamos al backend de PyTorch, que no toca numpy.
!pip install "tensorflow-cpu"

> Diagnóstico del entorno. Confirma qué backend y versión hay, si la GPU es visible, y las firmas exactas de las piezas de `ForecasterRnn` en esta versión de skforecast, para construir el modelo sin suposiciones.


In [ ]:
import os, inspect
import numpy as np
import skforecast
print("numpy      :", np.__version__)
print("skforecast :", skforecast.__version__)

# Backend de cálculo. Probamos TensorFlow; si no está, lo reportamos.
try:
    import tensorflow as tf
    print("tensorflow :", tf.__version__)
    gpus = tf.config.list_physical_devices('GPU')
    print("GPU (TF)   :", gpus if gpus else "ninguna, se usa CPU")
except Exception as e:
    print("tensorflow : NO disponible ->", repr(e))

import keras
print("keras      :", keras.__version__, "| backend:", keras.backend.backend())

# Firmas de las piezas clave de ForecasterRnn en skforecast 0.22
from skforecast.deep_learning import ForecasterRnn, create_and_compile_model
print("\ncreate_and_compile_model")
print("  ", inspect.signature(create_and_compile_model))
print("\nForecasterRnn.__init__")
print("  ", inspect.signature(ForecasterRnn.__init__))


print("
ForecasterRnn.fit")
print("  ", inspect.signature(ForecasterRnn.fit))

# Â¿Soporta exÃ³genas? Buscamos 'exog' en los mÃ©todos clave (decide si el LSTM puede llevar las 19 de RFECV)
print("
Â¿exog soportado?")
for meth in ("fit", "create_train_X_y", "predict"):
    try:
        sig = inspect.signature(getattr(ForecasterRnn, meth))
        print(f"   {meth:18s} exog={'exog' in sig.parameters}")
    except Exception as e:
        print(f"   {meth:18s} (sin introspecciÃ³n: {e})")


---
# <font color='#1B3A5C'>1. Carga de datos y preparacion multiserie</font>

> Se carga el mismo dataset que el resto de notebooks (`tfm_energy.dataset_ml` en MongoDB) y se prepara en el formato que necesita la red recurrente: la serie objetivo en formato ancho (una columna por barrio) y un unico bloque de exogenas compartidas.


In [ ]:
import os
os.environ['TQDM_DISABLE'] = '1'
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pymongo import MongoClient
import warnings, time

import keras
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from skforecast.deep_learning import ForecasterRnn, create_and_compile_model
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster_multiseries

warnings.filterwarnings('ignore')

plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#D3D3D3'
plt.rcParams['grid.linewidth'] = 0.4
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

C1 = '#264653'; C2 = '#2A9D8F'; C3 = '#E9C46A'
C4 = '#F4A261'; C5 = '#E76F51'; C6 = '#A8DADC'
TITULO = '#1B3A5C'; SUBTITULO = '#C0392B'

start_time = time.time()

> Carga del dataset desde MongoDB. Es la misma coleccion y el mismo dataframe que en los notebooks 06 y 07.


In [ ]:
client = MongoClient('mongodb://mongo:27017/')
db     = client['tfm_energy']

docs  = list(db['dataset_ml'].find({}, {'_id': 0}))
df_ml = pl.DataFrame(docs, infer_schema_length=None)

print(f"Shape: {df_ml.shape}")
print(f"Desde: {df_ml['datetime'].min()}  Hasta: {df_ml['datetime'].max()}")
print(f"Codigos postales: {df_ml['cod_postal'].n_unique()}")

> Constantes del estudio. Identicas a las del 06 y el 07 para que la comparacion sea limpia: mismo split, mismos horizontes y los mismos 30 codigos postales limpios (se excluyen los 12 corruptos de la auditoria del 04.5).


In [ ]:
INI       = '2019-01-01'
FIN_TRAIN = '2024-12-31 18:00:00'
FIN_VAL   = '2025-09-30 18:00:00'
INI_VAL   = '2025-01-01'
INI_TEST  = '2025-10-01'

HORIZONTES = {'24h': 4, '48h': 8, '72h': 12}
STEPS      = 12   # 72h en bloques de 6h

TARGET = 'mwh_total'
CPS_SUCIOS = ['08011','08009','08007','08013','08010','08006','08005',
              '08019','08008','08036','08026','08037']
CPS_TODOS = [cp for cp in sorted(df_ml['cod_postal'].unique().to_list()) if cp not in CPS_SUCIOS]

print(f"Train : {INI} a {FIN_TRAIN}")
print(f"Val   : {INI_VAL} a {FIN_VAL}")
print(f"Test  : {INI_TEST} a {df_ml['datetime'].max()}")
print(f"Target: {TARGET} | CPs limpios: {len(CPS_TODOS)} | steps: {STEPS}")

## <font color='#C0392B'>1.1 Serie objetivo en formato ancho</font>

> La red recurrente multiserie no recibe la tabla larga (una fila por barrio y hora), sino la serie en formato ancho: el indice es el tiempo (bloques de 6h) y cada columna es un barrio. Cada celda es el consumo de ese barrio en ese momento. Es la hoja de calculo que mira la red para predecir los 30 barrios a la vez.


In [ ]:
series_wide = (
    df_ml.filter(pl.col('cod_postal').is_in(CPS_TODOS))
         .select(['datetime', 'cod_postal', TARGET])
         .sort('datetime')
         .to_pandas()
         .pivot(index='datetime', columns='cod_postal', values=TARGET)
)
series_wide.index = pd.DatetimeIndex(series_wide.index)
series_wide = series_wide.asfreq('6h')

print(f"Serie ancha: {series_wide.shape[0]} bloques x {series_wide.shape[1]} barrios")
print(f"Rango: {series_wide.index.min()}  a  {series_wide.index.max()}")

> Auditoria de nulos en el target. A diferencia de los arboles, que digieren los NaN de forma nativa, las redes neuronales no los toleran: un solo NaN en la serie rompe el entrenamiento. Por eso primero se cuentan y luego se imputan.


In [ ]:
nan_cp = series_wide.isna().sum().sort_values(ascending=False)
print("NaN en el target por barrio (top 10):")
print(nan_cp.head(10))
print(f"\nBarrios con algun NaN: {(nan_cp > 0).sum()} de {series_wide.shape[1]}")
print(f"Total de NaN en el target: {int(series_wide.isna().sum().sum())}")
print(f"Porcentaje sobre el total de celdas: {series_wide.isna().sum().sum() / series_wide.size * 100:.3f}%")

## <font color='#C0392B'>1.2 Exogenas de ciudad (calendario + clima)</font>

> El `ForecasterRnn` admite un unico bloque de exogenas, comun a todos los barrios. Por eso se separan en dos grupos. El calendario (festivo, hora, dia, mes, semana, finde, covid) ya es identico en todos los barrios a una misma hora, asi que se toma tal cual. El clima varia por barrio (cada uno se asigna a una estacion meteorologica), de modo que se promedia entre los 30 barrios para obtener un clima de ciudad por cada momento.

 - Se deja fuera `anio`: en una red con escalado, el ano 2025 quedaria fuera del rango de entrenamiento (2019 a 2024) y el escalado lo mandaria por encima de 1, un valor no visto. El cambio de regimen de 2020 ya lo marca `is_covid`.
 - El clima fino por barrio se reserva para el plan B (un modelo por codigo postal).


In [ ]:
CAL = ['es_festivo', 'hora', 'dia_semana', 'mes', 'semana_anio', 'es_finde', 'is_covid']
MET = ['temp_mean', 'humedad_mean', 'viento_mean', 'irradiancia_mean', 'HDD', 'CDD']

# Calendario: identico entre barrios -> se toma de un barrio cualquiera
cal = (df_ml.filter(pl.col('cod_postal') == CPS_TODOS[0])
            .select(['datetime'] + CAL).sort('datetime')
            .to_pandas().set_index('datetime'))

# Clima de ciudad: promedio entre los 30 barrios limpios por cada momento
met = (df_ml.filter(pl.col('cod_postal').is_in(CPS_TODOS))
            .group_by('datetime').agg([pl.col(c).mean() for c in MET])
            .sort('datetime').to_pandas().set_index('datetime'))

exog_wide = cal.join(met)
exog_wide.index = pd.DatetimeIndex(exog_wide.index)
exog_wide = exog_wide.asfreq('6h')
for c in exog_wide.select_dtypes('bool').columns:
    exog_wide[c] = exog_wide[c].astype('int8')

print(f"Exogenas de ciudad: {exog_wide.shape[1]} variables")
print("Columnas:", list(exog_wide.columns))
print(f"\nNaN en exogenas: {int(exog_wide.isna().sum().sum())}")
print(exog_wide.isna().sum()[exog_wide.isna().sum() > 0])

## <font color='#C0392B'>1.3 Imputacion y particion</font>

> Imputacion de los NaN por interpolacion temporal (rellena el hueco siguiendo la pendiente entre el valor anterior y el posterior) y un relleno hacia delante y hacia atras para los bordes. Tras este paso ni el target ni las exogenas deben tener ningun NaN, requisito de la red. El escalado no se hace aqui a mano: lo gestiona internamente el `ForecasterRnn` mediante `transformer_series` y `transformer_exog` (un `MinMaxScaler`), de forma reversible en la prediccion.


In [ ]:
series_wide = series_wide.interpolate('time').bfill().ffill()
exog_wide   = exog_wide.interpolate('time').bfill().ffill()

print("NaN tras imputar:")
print(f"  target   : {int(series_wide.isna().sum().sum())}")
print(f"  exogenas : {int(exog_wide.isna().sum().sum())}")

# Particiones (mismas fechas que 06 y 07)
s_train, s_val, s_test = series_wide.loc[:FIN_TRAIN], series_wide.loc[INI_VAL:FIN_VAL], series_wide.loc[INI_TEST:]
e_train, e_val, e_test = exog_wide.loc[:FIN_TRAIN],   exog_wide.loc[INI_VAL:FIN_VAL],   exog_wide.loc[INI_TEST:]

print(f"\nTrain: {s_train.shape[0]} bloques ({s_train.index.min().date()} a {s_train.index.max().date()})")
print(f"Val  : {s_val.shape[0]} bloques ({s_val.index.min().date()} a {s_val.index.max().date()})")
print(f"Test : {s_test.shape[0]} bloques ({s_test.index.min().date()} a {s_test.index.max().date()})")